In [1]:
!pip install -q transformers peft bitsandbytes accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.0 MB/s eta 0:00:00


In [2]:
import os
import time
import torch
import shutil

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import PeftModel


print("========== GPU INFORMATION ==========")
print("GPU Name       :", torch.cuda.get_device_name(0))
print(
    "Total VRAM     :",
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    "GB"
)
print("=====================================")

========== GPU INFORMATION ==========
GPU Name       : Tesla T4
Total VRAM     : 15.64 GB


In [3]:
from google.colab import files
uploaded = files.upload()

Saving adapter_config.json to adapter_config.json
Saving adapter_model.safetensors to adapter_model.safetensors
Saving tokenizer_config.json to tokenizer_config.json


In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded successfully")
print(
    "Current GPU Memory Usage :",
    round(torch.cuda.memory_allocated() / 1e9, 2),
    "GB"
)

Initializing tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading base model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded successfully
Current GPU Memory Usage : 2.2 GB


In [6]:
print("Loading LoRA adapters into base model...")

model = PeftModel.from_pretrained(
    base_model,
    "/content/adapters"
)

print("Merging adapters with base model...")
model = model.merge_and_unload()

print("Model merge completed successfully")
print("Current GPU Memory Usage :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

Loading LoRA adapters into base model...
Merging adapters with base model...
Model merge completed successfully
Current GPU Memory Usage : 2.21 GB


In [7]:
print("Creating FP16 output directory...")

os.makedirs(
    "/content/quantized/model-fp16",
    exist_ok=True
)

print("Saving FP16 model...")
model.save_pretrained(
    "/content/quantized/model-fp16"
)

print("Saving tokenizer...")
tokenizer.save_pretrained(
    "/content/quantized/model-fp16"
)

print("FP16 model and tokenizer saved successfully")

Creating FP16 output directory...
Saving FP16 model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer...
FP16 model and tokenizer saved successfully


In [8]:
print("Configuring INT8 quantization...")

bnb_int8 = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading FP16 model for INT8 conversion...")
model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_int8,
    device_map="auto"
)

print("Creating INT8 output directory...")
os.makedirs(
    "/content/quantized/model-int8",
    exist_ok=True
)

print("Saving INT8 model...")
model_int8.save_pretrained(
    "/content/quantized/model-int8"
)

print("Saving tokenizer...")
tokenizer.save_pretrained(
    "/content/quantized/model-int8"
)

print("INT8 model saved successfully")

Configuring INT8 quantization...
Loading FP16 model for INT8 conversion...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Creating INT8 output directory...
Saving INT8 model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer...
INT8 model saved successfully


In [9]:
print("Configuring INT4 quantization...")

bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading FP16 model for INT4 conversion...")
model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_int4,
    device_map="auto"
)

print("Creating INT4 output directory...")
os.makedirs(
    "/content/quantized/model-int4",
    exist_ok=True
)

print("Saving INT4 model...")
model_int4.save_pretrained(
    "/content/quantized/model-int4"
)

print("Saving tokenizer...")
tokenizer.save_pretrained(
    "/content/quantized/model-int4"
)

print("INT4 model saved successfully")

Configuring INT4 quantization...
Loading FP16 model for INT4 conversion...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Creating INT4 output directory...
Saving INT4 model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer...
INT4 model saved successfully


In [10]:
print("Cloning llama.cpp repository...")
!git clone https://github.com/ggerganov/llama.cpp -q

print("Installing llama.cpp requirements...")
!pip install -q -r /content/llama.cpp/requirements.txt


from huggingface_hub import hf_hub_download


print("Downloading tokenizer.model from Hugging Face Hub...")
hf_hub_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    filename="tokenizer.model",
    local_dir="/content/quantized/model-fp16"
)

print("Converting Hugging Face model to GGUF format...")
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/quantized/model-fp16 \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0

print("GGUF file created successfully")
print(
    "GGUF File Size :",
    round(
        os.path.getsize("/content/quantized/model.gguf") / 1e9,
        2
    ),
    "GB"
)

Cloning llama.cpp repository...
Installing llama.cpp requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 482.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 106.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━

In [24]:
import os
import time
import torch
import subprocess
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

BASE_PATH = "/content/quantized"

PROMPT = "Explain hypertension and its complications."
MAX_NEW_TOKENS = 128


MODELS = {
    "FP16": {
        "path": f"{BASE_PATH}/model-fp16",
        "quant_config": None,
        "torch_dtype": torch.float16
    },
    "INT8": {
        "path": f"{BASE_PATH}/model-int8",
        "quant_config": BitsAndBytesConfig(
            load_in_8bit=True
        ),
        "torch_dtype": None
    },
    "INT4": {
        "path": f"{BASE_PATH}/model-int4",
        "quant_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        ),
        "torch_dtype": None
    }
}


def get_model_size(path):
    result = subprocess.check_output(["du", "-sh", path]).decode()
    return result.split()[0]


def benchmark_model(name, cfg):
    print(f"\n--- Benchmarking {name} ---")

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["path"],
        local_files_only=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        device_map="auto",
        local_files_only=True,
        torch_dtype=cfg["torch_dtype"],
        quantization_config=cfg["quant_config"]
    )

    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)

    torch.cuda.synchronize()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS
        )

    torch.cuda.synchronize()
    end = time.time()

    tokens_generated = output.shape[-1] - inputs["input_ids"].shape[-1]
    tokens_per_sec = tokens_generated / (end - start)

    size = get_model_size(cfg["path"])

    del model
    torch.cuda.empty_cache()

    return {
        "format": name,
        "size": size,
        "tokens_per_sec": round(tokens_per_sec, 2)
    }


results = []

for name, cfg in MODELS.items():
    res = benchmark_model(name, cfg)
    results.append(res)


print("\n===== BENCHMARK RESULTS =====")
for r in results:
    print(
        f"{r['format']:>5} | Size: {r['size']:>6} | Speed: {r['tokens_per_sec']} tokens/sec"
    )


--- Benchmarking FP16 ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Benchmarking INT8 ---


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Benchmarking INT4 ---


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  def supports_quant_method(quantization_config_dict):


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


===== BENCHMARK RESULTS =====
 FP16 | Size:   2.1G | Speed: 32.82 tokens/sec
 INT8 | Size:   1.2G | Speed: 9.02 tokens/sec
 INT4 | Size:   774M | Speed: 21.55 tokens/sec
